In [1]:
import pandas as pd
from pathlib import Path

PROCESSED = Path("../data/processed")

nodes = pd.read_csv(
    PROCESSED / "nodes.csv",
    usecols=["deployment_id", "platform", "in_catalogue"],
    dtype=str,
    keep_default_na=False,
)
edges = pd.read_csv(
    PROCESSED / "edges.csv",
    dtype={"source": str, "target": str},
    keep_default_na=False,
)

connected_ids = set(edges["source"]) | set(edges["target"])

print("Nodes:", len(nodes))
print("Connected nodes:", len(connected_ids))

Nodes: 86495
Connected nodes: 37239


In [2]:
pair_chunks = []

for chunk in pd.read_csv(
    PROCESSED / "votes_clean.csv",
    usecols=["deployment_id", "voter"],
    dtype=str,
    keep_default_na=False,
    chunksize=1_000_000,
):
    pair_chunks.append(chunk.drop_duplicates())

voter_dao_pairs = pd.concat(pair_chunks, ignore_index=True).drop_duplicates()
voter_dao_pairs = voter_dao_pairs[voter_dao_pairs["voter"] != ""]

voters_per_dao = voter_dao_pairs.groupby("deployment_id").size()

print("Voter-DAO pairs:", len(voter_dao_pairs))
print("Deployments with votes:", len(voters_per_dao))

Voter-DAO pairs: 2721671
Deployments with votes: 65752


In [3]:
def classify(deployment_id):
    if deployment_id in connected_ids:
        return "connected"
    if deployment_id in voters_per_dao.index:
        return "votes_no_edges"
    return "no_votes"

nodes["status"] = nodes["deployment_id"].map(classify)
nodes["voters_in_votes"] = (
    nodes["deployment_id"].map(voters_per_dao).fillna(0).astype(int)
)

print(nodes["status"].value_counts())

pd.crosstab(nodes["platform"], nodes["status"], margins=True, margins_name="total")

status
connected         37239
votes_no_edges    28513
no_votes          20743
Name: count, dtype: int64


status,connected,no_votes,votes_no_edges,total
platform,,,,
aragon,491,1408,503,2402
daohaus,769,2521,169,3459
daostack,30,3,25,58
realms,780,2771,673,4224
snapshot,34294,10795,26869,71958
tally,875,3245,274,4394
total,37239,20743,28513,86495


In [4]:
daos_per_voter = voter_dao_pairs.groupby("voter").size()
pairs = voter_dao_pairs.assign(
    voter_daos=voter_dao_pairs["voter"].map(daos_per_voter)
)

# Voters removed by edge_generator.py (more than 100 DAOs), per deployment
filtered_out = pairs[pairs["voter_daos"] > 100].groupby("deployment_id").size()
nodes["voters_filtered_out"] = (
    nodes["deployment_id"].map(filtered_out).fillna(0).astype(int)
)

isolated = nodes[nodes["status"] == "votes_no_edges"]
print("Isolated with votes:", len(isolated))
print("  only single-DAO voters:", (isolated["voters_filtered_out"] == 0).sum())
print("  isolated because of the filter:", (isolated["voters_filtered_out"] > 0).sum())

Isolated with votes: 28513
  only single-DAO voters: 28481
  isolated because of the filter: 32


In [5]:
bins = [0, 1, 5, 20, 100, float("inf")]
labels = ["1", "2-5", "6-20", "21-100", ">100"]

with_votes = nodes[nodes["voters_in_votes"] > 0].copy()
with_votes["size_band"] = pd.cut(with_votes["voters_in_votes"], bins=bins, labels=labels)

print(with_votes.groupby("status")["voters_in_votes"].median())
pd.crosstab(with_votes["size_band"], with_votes["status"], margins=True, margins_name="total")

status
connected         2.0
votes_no_edges    1.0
Name: voters_in_votes, dtype: float64


status,connected,votes_no_edges,total
size_band,,,
1,17382,24672,42054
2-5,10718,3185,13903
6-20,4034,518,4552
21-100,2925,106,3031
>100,2180,32,2212
total,37239,28513,65752


In [6]:
platform_of = dict(zip(nodes["deployment_id"], nodes["platform"]))
pd.crosstab(edges["source"].map(platform_of), edges["target"].map(platform_of))

target,aragon,daohaus,daostack,realms,snapshot,tally
source,,,,,,
aragon,472,265,24,0,3259,279
daohaus,269,5242,67,0,12291,1013
daostack,40,66,78,0,745,92
realms,0,0,0,3556,0,0
snapshot,3,18,0,0,468304,22356
tally,0,0,0,0,52798,5624


In [7]:
multi_voter = with_votes[with_votes["voters_in_votes"] >= 2]

print("Deployments with 2+ voters:", len(multi_voter))
pd.crosstab(
    multi_voter["platform"], multi_voter["status"], normalize="index"
).round(3)

Deployments with 2+ voters: 23698


status,connected,votes_no_edges
platform,,
aragon,0.663,0.337
daohaus,0.899,0.101
daostack,0.758,0.242
realms,0.611,0.389
snapshot,0.849,0.151
tally,0.856,0.144
